<a href="https://colab.research.google.com/github/AhmadZahran1011/BigData26_A_2411532004_AhmadZahran/blob/praktikum3/BD_A_P03_2411532004_AhmadZahran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

k-1


In [ ]:
import sys, time, os, json, sqlite3, gc
import pandas as pd, numpy as np, psutil

for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s} : {mod.__version__}")
    except ImportError:
        print(f"{nama:11s} : BELUM TERPASANG")

from google.colab import drive
drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum3"

for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

# Dipakai kembali dari Praktikum 1
proses = psutil.Process(os.getpid())
catatan = []

def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({
        "langkah": label,
        "detik": round(detik, 2),
        "delta_rss_mb": round(rss_mb() - m0, 1)
    })
    print(f"[{label}] {detik:.2f} s")
    return hasil
    # Baru di Praktikum 3: catatan asal-usul data (lineage)
lineage = []

def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama,
        "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris,
        "keterangan": keterangan
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")

pandas      : 2.2.3
numpy       : 2.1.3
pyarrow     : 23.0.1
requests    : 2.32.4
sklearn     : 1.6.1
sqlalchemy  : 2.0.54
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


k-2

In [ ]:
import requests, shutil

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada."""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan

    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    for potongan in r.iter_content(chunk_size=1024 * 1024):  # ikut dekompresi gzip
                        f.write(potongan)
            if os.path.getsize(sementara) == 0:
                raise IOError("berkas kosong")
            os.replace(sementara, tujuan)  # atomik
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i)
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)

    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

URL_TRIP = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

PATH_TRIP = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-01.parquet")
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")

# hapus cache yang rusak sebelum mengunduh ulang
if os.path.exists(PATH_ZONA):
    os.remove(PATH_ZONA)

ukur("unduh trip", lambda: unduh_aman(URL_TRIP, PATH_TRIP))
ukur("unduh zona", lambda: unduh_aman(URL_ZONA, PATH_ZONA))

for pth in (PATH_TRIP, PATH_ZONA):
    print(os.path.basename(pth), "->", round(os.path.getsize(pth)/1024**2, 2), "MB")

KOLOM = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
    "trip_distance", "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount"
]

trip = ukur("baca trip", lambda: pd.read_parquet(PATH_TRIP, columns=KOLOM))
zona = pd.read_csv(PATH_ZONA)

catat_sumber("trip", URL_TRIP, len(trip), "Parquet bulanan TLC")
catat_sumber("zona", URL_ZONA, len(zona), "tabel dimensi 265 zona")

zona.head()

cache ditemukan: yellow_tripdata_2023-01.parquet
[unduh trip] 0.00 s
[unduh zona] 0.04 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
taxi_zone_lookup.csv -> 0.01 MB
[baca trip] 0.68 s
[lineage] trip: 3,066,766 baris
[lineage] zona: 265 baris


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


k-3

In [ ]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }
    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data: # validasi struktur
                raise ValueError("kunci 'hourly' tidak ada pada respons")
            return pd.DataFrame(data["hourly"])
        if r.status_code in (429, 500, 502, 503): # layak dicoba ulang
            time.sleep(2**i)
            continue
        r.raise_for_status() # galat lain: hentikan
    raise RuntimeError("API cuaca tidak merespons dengan benar")

cuaca = ukur("ambil cuaca", lambda: ambil_cuaca("2023-01-01", "2023-01-31"))
cuaca["time"] = pd.to_datetime(cuaca["time"])
cuaca = cuaca.rename(columns={
    "time": "jam_mulai",
    "temperature_2m": "suhu_c",
    "precipitation": "hujan_mm"
})

catat_sumber("cuaca", API_CUACA, len(cuaca), "Open-Meteo hourly archive")
cuaca.head()

[ambil cuaca] 0.65 s
[lineage] cuaca: 744 baris


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.9,1.0
1,2023-01-01 01:00:00,10.6,1.0
2,2023-01-01 02:00:00,10.6,0.1
3,2023-01-01 03:00:00,10.5,0.0
4,2023-01-01 04:00:00,9.8,0.0


k-4

k-5

In [ ]:
from sqlalchemy import create_engine

PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")
engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis",
                       "Sengketa", "Tidak diketahui", "Perjalanan batal"],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})

tarif_referensi.to_sql("tarif_referensi", engine, if_exists="replace", index=False)

# Tabel transaksi tiruan untuk latihan pembacaan bertahap
trip.head(300_000).to_sql(
    "trip_operasional", engine, if_exists="replace",
    index=False, chunksize=50_000
)

print("Tabel dibuat:", pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", engine)["name"].tolist())

SQL = """
SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
FROM trip_operasional
WHERE total_amount > 0
GROUP BY payment_type
ORDER BY jumlah DESC
"""

ringkas_db = pd.read_sql(SQL, engine)
print(ringkas_db)

# Pembacaan bertahap: agregasi tanpa memuat seluruh tabel ke memori
total_baris = 0
for bagian in pd.read_sql("SELECT trip_distance FROM trip_operasional",
                          engine, chunksize=50_000):
    total_baris += len(bagian)

print("dibaca bertahap:", f"{total_baris:,}", "baris")

tarif_referensi = pd.read_sql("SELECT * FROM tarif_referensi", engine)
catat_sumber("tarif_referensi", PATH_DB, len(tarif_referensi),
             "tabel dimensi dari basis data operasional")

Tabel dibuat: ['tarif_referensi', 'trip_operasional']
   payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691
dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


k-6

In [ ]:
trip["durasi_menit"] = (
    (trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
)

def profil_kolom(df):
    baris = []
    for kol in df.columns:
        s = df[kol]
        baris.append({
            "kolom": kol, "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(baris)

profil = profil_kolom(trip)
print(profil)

AWAL, AKHIR = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-02-01")
zona_sah = set(zona["LocationID"])

aturan = {
    "kelengkapan: passenger_count ada": trip["passenger_count"].notna(),
    "keunikan: baris tidak duplikat": ~trip.duplicated(),
    "validitas: jarak 0-100 mil": trip["trip_distance"].between(0.01, 100),
    "validitas: total_amount > 0": trip["total_amount"] > 0,
    "validitas: PULocationID dikenal": trip["PULocationID"].isin(zona_sah),
    "konsistensi: durasi 1-180 menit": trip["durasi_menit"].between(1, 180),
    "konsistensi: total >= fare": trip["total_amount"] >= trip["fare_amount"],
    "ketepatan waktu dalam Jan 2023": trip["tpep_pickup_datetime"].between(AWAL, AKHIR),
}

laporan = pd.DataFrame([
    {
        "aturan": nama,
        "lulus": int(mask.sum()),
        "gagal": int((~mask).sum()),
        "persen_gagal": round(100 * (~mask).mean(), 3)
    }
    for nama, mask in aturan.items()
]).sort_values("persen_gagal", ascending=False)

laporan

                    kolom            tipe  persen_hilang  nilai_unik  \
0    tpep_pickup_datetime  datetime64[us]          0.000     1610975   
1   tpep_dropoff_datetime  datetime64[us]          0.000     1611319   
2         passenger_count         float64          2.339          10   
3           trip_distance         float64          0.000        4387   
4            PULocationID           int64          0.000         257   
5            DOLocationID           int64          0.000         261   
6            payment_type           int64          0.000           5   
7             fare_amount         float64          0.000        6873   
8              tip_amount         float64          0.000        4036   
9            total_amount         float64          0.000       15871   
10           durasi_menit         float64          0.000        8579   

                 contoh  
0   2023-01-01 00:32:10  
1   2023-01-01 00:40:36  
2                   1.0  
3                  0.97  
4    

,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
2,validitas: jarak 0-100 mil,3020816,45950,1.498
5,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
6,konsistensi: total >= fare,3041743,25023,0.816
7,ketepatan waktu dalam Jan 2023,3066718,48,0.002
1,keunikan: baris tidak duplikat,3066766,0,0.000
4,validitas: PULocationID dikenal,3066766,0,0.000


k-7

In [ ]:
KOL = "passenger_count"
asli = trip[KOL]
print("persen hilang:", round(100 * asli.isna().mean(), 3))

trip["jam"] = trip["tpep_pickup_datetime"].dt.hour

strategi = {
    "1_dibuang": asli.dropna(),
    "2_isi_median": asli.fillna(asli.median()),
    "3_isi_modus": asli.fillna(asli.mode().iloc[0]),
    "4_median_perjam": asli.fillna(trip.groupby("jam")[KOL].transform("median")),
}

banding = pd.DataFrame([
    {
        "strategi": nama,
        "n": len(s),
        "mean": round(s.mean(), 4),
        "median": s.median(),
        "std": round(s.std(), 4)
    }
    for nama, s in strategi.items()
])

print(banding)

# Strategi yang dipilih: tandai + isi per kelompok
trip[KOL + "_hilang"] = trip[KOL].isna().astype("int8")
trip[KOL] = trip[KOL].fillna(trip.groupby("jam")[KOL].transform("median"))
trip[KOL] = trip[KOL].fillna(trip[KOL].median()) # jaring pengaman

# Duplikat: periksa dua tingkat
dup_penuh = trip.duplicated().sum()
KUNCI = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "total_amount"
]
dup_kunci = trip.duplicated(subset=KUNCI).sum()

print(f"duplikat penuh: {dup_penuh:,} | duplikat pada kunci: {dup_kunci:,}")

sebelum = len(trip)
trip = trip.drop_duplicates(subset=KUNCI, keep="first").reset_index(drop=True)
print(f"{sebelum:,} -> {len(trip):,} baris")

persen hilang: 2.339
          strategi        n    mean  median     std
0        1_dibuang  2995023  1.3625     1.0  0.8961
1     2_isi_median  3066766  1.3541     1.0  0.8873
2      3_isi_modus  3066766  1.3541     1.0  0.8873
3  4_median_perjam  3066766  1.3541     1.0  0.8873
duplikat penuh: 0 | duplikat pada kunci: 1
3,066,766 -> 3,066,765 baris


k-8

In [ ]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def ringkas_outlier(df, kolom):
    hasil = []
    for kol in kolom:
        s = df[kol].dropna()
        lo, hi = batas_iqr(s)
        z = (s - s.mean()) / s.std()
        hasil.append({
            "kolom": kol,
            "batas_bawah": round(lo, 2),
            "batas_atas": round(hi, 2),
            "outlier_iqr": int(((s < lo) | (s > hi)).sum()),
            "outlier_z3": int((z.abs() > 3).sum()),
            "p99": round(s.quantile(0.99), 2),
        })
    return pd.DataFrame(hasil)

print(ringkas_outlier(trip, ["trip_distance", "durasi_menit", "total_amount"]))

# Filter data yang layak
layak = (
    trip["trip_distance"].between(0.01, 100)
    & trip["durasi_menit"].between(1, 180)
    & (trip["total_amount"] > 0)
)
bersih = trip.loc[layak].copy()

# Tandai outlier tarif ekstrem
lo, hi = batas_iqr(bersih["total_amount"])
bersih["tarif_ekstrem"] = (bersih["total_amount"] > hi).astype("int8")

# Winsorizing trip_distance
batas_atas = bersih["trip_distance"].quantile(0.995)
bersih["trip_distance_capped"] = bersih["trip_distance"].clip(upper=batas_atas)

print(f"{len(trip):,} -> {len(bersih):,} baris ({100*(1-len(bersih)/len(trip)):.2f}% dibuang)")

# Join 1: tabel zona (dimensi)
print("kunci zona unik?", zona["LocationID"].is_unique) # WAJIB diperiksa
n0 = len(bersih)
bersih = bersih.merge(
    zona[["LocationID", "Borough", "Zone"]]
    .rename(columns={"Borough": "borough_naik", "Zone": "zona_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left", validate="many_to_one"
)
bersih = bersih.drop(columns="LocationID")
print(f"baris {n0:,} -> {len(bersih):,} (harus sama)")
print("zona tak dikenal:", bersih["zona_naik"].isna().sum())

# Join 2: tabel pembayaran dari basis data
bersih = bersih.merge(
    tarif_referensi[["payment_type", "nama_pembayaran"]],
    on="payment_type", how="left", validate="many_to_one"
)

# Join 3: cuaca, berdasarkan jam
bersih["jam_mulai"] = bersih["tpep_pickup_datetime"].dt.floor("h")
n0 = len(bersih)
bersih = bersih.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one")
print(f"baris {n0:,} -> {len(bersih):,}")
print("tanpa data cuaca:", bersih["suhu_c"].isna().sum())

           kolom  batas_bawah  batas_atas  outlier_iqr  outlier_z3     p99
0  trip_distance        -2.35        6.74       390244          67   20.06
1   durasi_menit        -9.66       35.08       170615        3175   57.25
2   total_amount        -4.55       48.65       371616       87248  101.94
3,066,765 -> 2,986,910 baris (2.60% dibuang)
kunci zona unik? True
baris 2,986,910 -> 2,986,910 (harus sama)
zona tak dikenal: 37609
baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


k-9

In [ ]:
bersih["hari_minggu"] = bersih["tpep_pickup_datetime"].dt.dayofweek
bersih["akhir_pekan"] = (bersih["hari_minggu"] >= 5).astype("int8")
bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
FITUR_KAT = ["nama_pembayaran", "borough_naik"]

contoh = bersih.dropna(subset=FITUR_NUM + FITUR_KAT).sample(
    n=min(200_000, len(bersih)), random_state=42
)

latih, uji = train_test_split(contoh, test_size=0.2, random_state=42)

# StandardScaler
skala = StandardScaler().fit(latih[FITUR_NUM]) # fit HANYA pada data latih
latih_num = skala.transform(latih[FITUR_NUM])
uji_num = skala.transform(uji[FITUR_NUM]) # transform saja

print("mean latih (harus ~0):", latih_num.mean(axis=0).round(3))
print("mean uji (tidak harus 0):", uji_num.mean(axis=0).round(3))

# OneHotEncoder
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(latih[FITUR_KAT])
latih_kat = enc.transform(latih[FITUR_KAT])

print("jumlah kolom hasil one-hot:", latih_kat.shape[1])
print("nama kolom:", enc.get_feature_names_out()[:6], "...")

mean latih (harus ~0): [-0.  0. -0.]
mean uji (tidak harus 0): [ 0.001  0.001 -0.009]
jumlah kolom hasil one-hot: 11
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tunai' 'borough_naik_Bronx'
 'borough_naik_Brooklyn'] ...


k-10

In [ ]:
KONTRAK = {
    "tpep_pickup_datetime": "datetime64[ns]",
    "trip_distance": "float64",
    "durasi_menit": "float64",
    "total_amount": "float64",
    "borough_naik": "object",
    "nama_pembayaran": "object",
}

def validasi(df, kontrak=KONTRAK):
    masalah = []
    for kol, tipe in kontrak.items():
        if kol not in df.columns:
            masalah.append(f"kolom hilang: {kol}")
        elif not str(df[kol].dtype).startswith(tipe.split("[")[0]):
            masalah.append(f"tipe {kol}: {df[kol].dtype} != {tipe}")
    if df.empty:
        masalah.append("DataFrame kosong")
    if len(df) != len(df.drop_duplicates(subset=KUNCI)):
        masalah.append("masih ada duplikat pada kunci")
    if masalah:
        raise AssertionError("Validasi gagal:\n- " + "\n- ".join(masalah))
    print(f"Validasi lulus: {len(df):,} baris x {df.shape[1]} kolom")
    return df

validasi(bersih)

def pipeline(path_trip, path_zona, df_cuaca, df_tarif):
    """Satu fungsi, idempoten: input mentah -> DataFrame tervalidasi."""
    df = pd.read_parquet(path_trip, columns=KOLOM)
    z = pd.read_csv(path_zona)

    df["durasi_menit"] = ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
                          .dt.total_seconds() / 60)
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median"))

    df = df.drop_duplicates(subset=KUNCI)
    df = df[df["trip_distance"].between(0.01, 100)
            & df["durasi_menit"].between(1, 180)
            & (df["total_amount"] > 0)]

    df = (df.merge(z[["LocationID", "Borough"]]
                   .rename(columns={"Borough": "borough_naik"}),
                   left_on="PULocationID", right_on="LocationID",
                   how="left", validate="many_to_one")
          .drop(columns="LocationID")
          .merge(df_tarif[["payment_type", "nama_pembayaran"]],
                 on="payment_type", how="left", validate="many_to_one"))

    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(df_cuaca, on="jam_mulai", how="left", validate="many_to_one")

    return validasi(df)

hasil = ukur("pipeline penuh", lambda: pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi))

OUT = os.path.join(DIR_KURASI, "trips_bersih")
hasil["tanggal"] = hasil["tpep_pickup_datetime"].dt.date.astype(str)

ukur("tulis parquet berpartisi", lambda: hasil.to_parquet(
    OUT, partition_cols=["borough_naik"], index=False))

print("partisi yang terbentuk:", sorted(os.listdir(OUT))[:5], "...")

# Bukti idempotensi: jalankan ulang, hasil harus identik
ulang = pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi)
print("idempoten?", len(ulang) == len(hasil))

# Artefak yang dikumpulkan
laporan.to_csv(f"{DIR_SIMPAN}/laporan_kualitas.csv", index=False)
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih", "zip", OUT)

Validasi lulus: 2,986,910 baris x 26 kolom
Validasi lulus: 2,986,910 baris x 17 kolom
[pipeline penuh] 14.35 s
[tulis parquet berpartisi] 4.12 s
partisi yang terbentuk: ['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens'] ...
Validasi lulus: 2,986,910 baris x 17 kolom
idempoten? True


'/content/drive/MyDrive/BigData/Praktikum3/trips_bersih.zip'

k-11

In [ ]:
!pip install -q pyspark==3.5.1

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("BD-P02").master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

sdf = spark.read.parquet(PATH_TRIP).select(*KOLOM)
szona = spark.createDataFrame(zona[["LocationID", "Borough"]])

sbersih = (sdf
    .withColumn("durasi_menit",
        (F.unix_timestamp(F.col("tpep_dropoff_datetime"))
         - F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 60)
    .filter(F.col("trip_distance").between(0.01, 100))
    .filter(F.col("durasi_menit").between(1, 180))
    .filter(F.col("total_amount") > 0)
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                     "PULocationID", "DOLocationID", "total_amount"])
    .join(F.broadcast(szona), # tabel kecil: broadcast
          sdf.PULocationID == szona.LocationID, "left")
    .drop("LocationID"))

ukur("spark: hitung baris bersih", lambda: sbersih.count())

sbersih.groupBy("Borough").count().orderBy(F.desc("count")).show(10, False)

sbersih.write.mode("overwrite").partitionBy("Borough") \
    .parquet("/content/lapisan_terkurasi/trips_spark")

sbersih.explain() # cari BroadcastHashJoin, bukan SortMergeJoin
spark.stop()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
[spark: hitung baris bersih] 14.78 s
+-------------+-------+
|Borough      |count  |
+-------------+-------+
|Manhattan    |2659609|
|Queens       |270315 |
|Unknown      |37609  |
|Brooklyn     |15724  |
|Bronx        |3074   |
|NaN          |340    |
|Staten Island|211    |
|EWR          |28     |
+-------------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#245, trip_distance#247, PULocationID#7L, DOLocatio

STUDI KASUS

1.Tabel Analitik

In [ ]:
tabel_analitik = (
    bersih
    .assign(
        jam_mulai=bersih["tpep_pickup_datetime"].dt.floor("h")
    )
    .groupby(
        ["borough_naik", "jam_mulai"],
        observed=True
    )
    .agg(
        jumlah_perjalanan=("total_amount", "size"),
        rata_tarif_per_mil=("tarif_per_mil", "median"),
        rata_kecepatan=("kecepatan_mph", "median"),
        suhu_c=("suhu_c", "first"),
        hujan=("hujan", "max")
    )
    .reset_index()
)

# Hanya mempertahankan kelompok dengan minimal 30 perjalanan
tabel_analitik = tabel_analitik[
    tabel_analitik["jumlah_perjalanan"] >= 30
].copy()

print("Ukuran tabel analitik:", tabel_analitik.shape)
tabel_analitik.head()

Ukuran tabel analitik: (2094, 7)


,borough_naik,jam_mulai,jumlah_perjalanan,rata_tarif_per_mil,rata_kecepatan,suhu_c,hujan
626,Brooklyn,2023-01-01 00:00:00,86,6.9385,14.550,10.9,1
627,Brooklyn,2023-01-01 01:00:00,220,6.9820,13.715,10.6,1
628,Brooklyn,2023-01-01 02:00:00,256,6.6215,15.445,10.6,0
629,Brooklyn,2023-01-01 03:00:00,175,6.6160,16.390,10.5,0
630,Brooklyn,2023-01-01 04:00:00,74,6.9120,14.635,9.8,0


### Hasil

Tabel analitik menghasilkan **2.094 baris dan 7 kolom**. Setiap baris mewakili kombinasi `borough_naik` dan `jam_mulai` yang memiliki minimal 30 perjalanan.

Kolom yang dihasilkan mencakup jumlah perjalanan, median tarif per mil, median kecepatan, suhu, dan indikator hujan. Contoh pada data Brooklyn menunjukkan bahwa nilai tarif per mil dan kecepatan berbeda pada setiap jam, sedangkan `hujan` bernilai 1 atau 0 sesuai kondisi hujan pada jam tersebut.

2.Angka Pembanding Hujan vs tidak hujan per borough

In [ ]:
# Perbandingan tarif per mil saat hujan dan tidak hujan
perbandingan_pricing = (
    tabel_analitik
    .groupby(
        ["borough_naik", "hujan"],
        observed=True
    )["rata_tarif_per_mil"]
    .median()
    .unstack()
    .rename(columns={
        False: "tidak_hujan",
        True: "hujan"
    })
)

perbandingan_pricing["selisih"] = (
    perbandingan_pricing["hujan"]
    - perbandingan_pricing["tidak_hujan"]
)

perbandingan_pricing["persen_perubahan"] = (
    perbandingan_pricing["selisih"]
    / perbandingan_pricing["tidak_hujan"]
    * 100
)

print(perbandingan_pricing.round(3))

hujan         tidak_hujan   hujan  selisih  persen_perubahan
borough_naik                                                
Brooklyn            6.943   7.655    0.712            10.255
Manhattan          10.527  11.400    0.873             8.293
Queens              5.453   5.636    0.183             3.365
Unknown             8.726   9.886    1.160            13.300


### Hasil Perbandingan Tarif Saat Hujan dan Tidak Hujan

Hasil perbandingan menunjukkan bahwa median tarif per mil lebih tinggi pada kondisi hujan di seluruh borough yang memiliki data pembanding. Brooklyn mengalami kenaikan dari 6.943 menjadi 7.655 atau sebesar 10.255%. Manhattan meningkat dari 10.527 menjadi 11.400 atau 8.293%. Queens meningkat dari 5.453 menjadi 5.636 atau 3.365%, sedangkan kelompok Unknown meningkat dari 8.726 menjadi 9.886 atau 13.300%.

Secara deskriptif, hasil ini menunjukkan adanya hubungan antara kondisi hujan dan tarif per mil yang lebih tinggi. Namun, hasil ini belum membuktikan bahwa hujan secara langsung menyebabkan kenaikan tarif karena faktor lain, seperti jam sibuk, dapat terjadi bersamaan dengan hujan.

Hipotesis: “Tarif per mile rises during rainy hours in certain boroughs.”

Hasil data mendukung hipotesis secara deskriptif. Pada Brooklyn, Manhattan, Queens, dan kelompok Unknown, median tarif per mil saat hujan lebih tinggi dibandingkan saat tidak hujan. Kenaikannya masing-masing sebesar 10.255%, 8.293%, 3.365%, dan 13.300%.

3. Batasan Eksplisit
Median digunakan sebagai ukuran pemusatan, bukan rerata, karena distribusi data cenderung miring dan dapat dipengaruhi oleh nilai ekstrem (outlier).
Data cuaca hanya diukur pada satu titik pengamatan yang dianggap mewakili kondisi seluruh kota, sehingga kemungkinan terdapat perbedaan kondisi cuaca di wilayah lain.
Korelasi tidak menunjukkan hubungan sebab-akibat. Hubungan antara curah hujan dan variabel lainnya tidak dapat langsung diartikan bahwa hujan menyebabkan perubahan tersebut.
Jam pengamatan dapat menjadi faktor perancu, karena hujan dapat terjadi bersamaan dengan jam sibuk sehingga perubahan kondisi yang diamati belum tentu sepenuhnya disebabkan oleh hujan.
Kualitas dan kelengkapan data bergantung pada sumber data yang digunakan. Data yang hilang, kesalahan pencatatan, atau perbedaan metode pengukuran antarbulan dapat memengaruhi hasil analisis.


4. Pembaruan Tabel Secara Otomatis Setiap Bulan
Agar tabel dapat diperbarui secara otomatis setiap bulan, terlebih dahulu perlu dibuat sumber data yang terstruktur dan konsisten, misalnya database atau file CSV/Excel dengan format kolom yang tetap. Selanjutnya, proses pengambilan data (data collection) dijadwalkan setiap awal atau akhir bulan menggunakan API, web scraping, atau sumber data resmi yang tersedia. Data baru kemudian masuk ke proses pembersihan dan validasi, seperti menangani data kosong, menghapus duplikasi, memeriksa kesalahan format, dan menghitung nilai median sesuai metode analisis yang digunakan. Setelah itu, sistem menjalankan proses perhitungan dan memperbarui tabel secara otomatis, misalnya menggunakan Python, SQL, atau spreadsheet dengan koneksi data. Terakhir, hasil pembaruan dapat disimpan dan ditampilkan pada tabel atau dashboard, serta dibuat jadwal otomatis (scheduled task/cron job) agar seluruh proses berjalan tanpa harus dilakukan secara manual setiap bulan.

LATIHAN

1.

In [ ]:
import time

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry, penulisan atomik, dan informasi kecepatan."""

    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        print("tidak ada unduhan ulang")
        return tujuan

    sementara = tujuan + ".part"

    for i in range(percobaan):
        try:
            t0 = time.perf_counter()

            with requests.get(
                url,
                stream=True,
                timeout=60
            ) as r:
                r.raise_for_status()

                with open(sementara, "wb") as f:
                    shutil.copyfileobj(r.raw, f)

            ukuran_mb = os.path.getsize(sementara) / 1024**2

            if ukuran_mb == 0:
                raise IOError("berkas kosong")

            os.replace(sementara, tujuan)

            detik = time.perf_counter() - t0
            kecepatan = ukuran_mb / detik

            print(
                f"unduh selesai: {ukuran_mb:.2f} MB "
                f"dalam {detik:.2f} detik"
            )
            print(
                f"kecepatan unduh: {kecepatan:.2f} MB/detik"
            )

            return tujuan

        except Exception as e:
            jeda = jeda_awal * (2 ** i)
            print(
                f"percobaan {i+1} gagal ({e}); "
                f"menunggu {jeda}s"
            )
            time.sleep(jeda)

    raise RuntimeError(
        f"gagal mengunduh setelah {percobaan} percobaan: {url}"
    )


# Gunakan file khusus untuk membuktikan dua kondisi
PATH_LATIHAN1 = os.path.join(
    DIR_MENTAH,
    "latihan1_yellow_tripdata.parquet"
)

# Hapus file uji jika sebelumnya sudah ada
if os.path.exists(PATH_LATIHAN1):
    os.remove(PATH_LATIHAN1)

print("=== Panggilan pertama ===")
unduh_aman(
    URL_TRIP,
    PATH_LATIHAN1
)

print("\n=== Panggilan kedua ===")
unduh_aman(
    URL_TRIP,
    PATH_LATIHAN1
)

=== Panggilan pertama ===
unduh selesai: 45.46 MB dalam 0.44 detik
kecepatan unduh: 103.92 MB/detik

=== Panggilan kedua ===
cache ditemukan: latihan1_yellow_tripdata.parquet
tidak ada unduhan ulang


'/content/lapisan_mentah/latihan1_yellow_tripdata.parquet'

2.

In [ ]:
aturan_q2 = aturan.copy()

# Aturan baru 1: validitas tip_amount
aturan_q2["validitas: tip_amount >= 0"] = (
    trip["tip_amount"] >= 0
)

# Aturan baru 2: konsistensi total dengan fare + tip
aturan_q2["konsistensi: total >= fare + tip"] = (
    trip["total_amount"]
    >= trip["fare_amount"] + trip["tip_amount"]
)

laporan_q2 = pd.DataFrame([
    {
        "aturan": nama,
        "lulus": int(mask.sum()),
        "gagal": int((~mask).sum()),
        "persen_gagal": round(
            100 * (~mask).mean(),
            3
        )
    }
    for nama, mask in aturan_q2.items()
]).sort_values(
    "persen_gagal",
    ascending=False
)

laporan_q2

,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
2,validitas: jarak 0-100 mil,3020816,45950,1.498
5,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
9,konsistensi: total >= fare + tip,3041605,25160,0.820
6,konsistensi: total >= fare,3041743,25023,0.816
8,validitas: tip_amount >= 0,3066540,225,0.007
7,ketepatan waktu dalam Jan 2023,3066718,48,0.002
1,keunikan: baris tidak duplikat,3066766,0,0.000
4,validitas: PULocationID dikenal,3066766,0,0.000


3.

In [ ]:
# Tabel zona tujuan
zona_tujuan = (
    zona[["LocationID", "Borough"]]
    .rename(columns={
        "LocationID": "DOLocationID",
        "Borough": "borough_tujuan"
    })
)

# Join lokasi tujuan
bersih_tujuan = bersih.merge(
    zona_tujuan,
    on="DOLocationID",
    how="left",
    validate="many_to_one"
)

# Hitung pasangan borough asal-tujuan
top10_pasangan = (
    bersih_tujuan
    .groupby(
        ["borough_naik", "borough_tujuan"],
        dropna=False
    )
    .size()
    .reset_index(name="jumlah_perjalanan")
    .sort_values(
        "jumlah_perjalanan",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

top10_pasangan

,borough_naik,borough_tujuan,jumlah_perjalanan
0,Manhattan,Manhattan,2491566
1,Queens,Manhattan,155261
2,Manhattan,Queens,82354
3,Manhattan,Brooklyn,62986
4,Queens,Queens,59146
5,Queens,Brooklyn,42187
6,Unknown,Manhattan,18432
7,Unknown,Unknown,13562
8,Manhattan,Bronx,8865
9,Brooklyn,Brooklyn,7910


4.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler
scaler_standard = StandardScaler().fit(
    latih[FITUR_NUM]
)

standard_num = scaler_standard.transform(
    latih[FITUR_NUM]
)

# MinMaxScaler
scaler_minmax = MinMaxScaler().fit(
    latih[FITUR_NUM]
)

minmax_num = scaler_minmax.transform(
    latih[FITUR_NUM]
)

# Ringkasan rentang hasil
perbandingan_scaling = pd.DataFrame({
    "fitur": FITUR_NUM,

    "standard_min": standard_num.min(axis=0),
    "standard_max": standard_num.max(axis=0),

    "minmax_min": minmax_num.min(axis=0),
    "minmax_max": minmax_num.max(axis=0)
})

perbandingan_scaling.round(3)

,fitur,standard_min,standard_max,minmax_min,minmax_max
0,trip_distance_capped,-0.794,4.247,0.0,1.0
1,durasi_menit,-1.226,13.925,0.0,1.0
2,suhu_c,-2.288,3.347,0.0,1.0


5.

In [ ]:
from pandas.errors import MergeError

# Simpan kondisi asli
zona_asli = zona.copy(deep=True)

# Buat tabel uji dengan satu LocationID duplikat
zona_uji = pd.concat(
    [
        zona.copy(),
        zona.iloc[[0]].copy()
    ],
    ignore_index=True
)

print(
    "LocationID unik sebelum:",
    zona["LocationID"].is_unique
)

print(
    "LocationID unik setelah digandakan:",
    zona_uji["LocationID"].is_unique
)

try:
    uji_many_to_one = bersih.merge(
        zona_uji[
            ["LocationID", "Borough"]
        ].rename(
            columns={
                "Borough": "borough_uji"
            }
        ),
        left_on="PULocationID",
        right_on="LocationID",
        how="left",
        validate="many_to_one"
    )

except MergeError as e:
    print("ERROR BERHASIL DITANGKAP:")
    print(type(e).__name__)
    print(e)

# Kembalikan tabel zona ke kondisi semula
zona = zona_asli.copy(deep=True)

print(
    "\nLocationID unik setelah dipulihkan:",
    zona["LocationID"].is_unique
)

LocationID unik sebelum: True
LocationID unik setelah digandakan: False
ERROR BERHASIL DITANGKAP:
MergeError
Merge keys are not unique in right dataset; not a many-to-one merge

LocationID unik setelah dipulihkan: True


6.

In [ ]:
import calendar

DIR_INCREMENTAL = os.path.join(
    DIR_KURASI,
    "trips_bersih_incremental"
)

os.makedirs(
    DIR_INCREMENTAL,
    exist_ok=True
)


def pipeline_inkremental(bulan):
    """
    Memproses satu bulan hanya jika output bulan tersebut
    belum tersedia.

    Format bulan: YYYY-MM
    """

    periode = pd.Period(
        bulan,
        freq="M"
    )

    bulan_str = str(periode)

    # Direktori output khusus bulan
    out_bulan = os.path.join(
        DIR_INCREMENTAL,
        bulan_str
    )

    # Cek apakah bulan sudah pernah diproses
    if os.path.exists(out_bulan) and os.listdir(out_bulan):
        print(
            f"[SKIP] {bulan_str} sudah ada di output."
        )
        print(
            "Tidak ada berkas baru yang dibuat."
        )
        return False

    os.makedirs(
        out_bulan,
        exist_ok=True
    )

    # Path data trip bulan tersebut
    nama_file = (
        f"yellow_tripdata_{bulan_str}.parquet"
    )

    path_trip_bulan = os.path.join(
        DIR_MENTAH,
        nama_file
    )

    url_trip_bulan = (
        "https://d37ci6vzurychx.cloudfront.net/"
        "trip-data/"
        f"{nama_file}"
    )

    # Unduh data trip
    unduh_aman(
        url_trip_bulan,
        path_trip_bulan
    )

    # Tentukan tanggal awal dan akhir bulan
    tanggal_awal = periode.start_time.strftime(
        "%Y-%m-%d"
    )
    tanggal_akhir = periode.end_time.strftime(
        "%Y-%m-%d"
    )

    # Ambil data cuaca untuk bulan tersebut
    cuaca_bulan = ambil_cuaca(
        tanggal_awal,
        tanggal_akhir
    )

    cuaca_bulan["time"] = pd.to_datetime(
        cuaca_bulan["time"]
    )

    cuaca_bulan = cuaca_bulan.rename(
        columns={
            "time": "jam_mulai",
            "temperature_2m": "suhu_c",
            "precipitation": "hujan_mm"
        }
    )

    # Jalankan pipeline K-10
    hasil_bulan = pipeline(
        path_trip_bulan,
        PATH_ZONA,
        cuaca_bulan,
        tarif_referensi
    )

    # Simpan output bulan
    hasil_bulan["tanggal"] = (
        hasil_bulan[
            "tpep_pickup_datetime"
        ]
        .dt.date
        .astype(str)
    )

    hasil_bulan.to_parquet(
        out_bulan,
        partition_cols=["borough_naik"],
        index=False
    )

    print(
        f"[SELESAI] {bulan_str}: "
        f"{len(hasil_bulan):,} baris"
    )

    print(
        "Output:",
        out_bulan
    )

    return True

In [ ]:
# Pastikan bulan uji belum ada
OUT_UJI = os.path.join(
    DIR_INCREMENTAL,
    "2023-01"
)

if os.path.exists(OUT_UJI):
    shutil.rmtree(OUT_UJI)

print("=== Panggilan pertama ===")

hasil_pertama = pipeline_inkremental(
    "2023-01"
)

jumlah_file_pertama = sum(
    len(files)
    for _, _, files in os.walk(OUT_UJI)
)

print(
    "Jumlah file setelah panggilan pertama:",
    jumlah_file_pertama
)

print("\n=== Panggilan kedua ===")

hasil_kedua = pipeline_inkremental(
    "2023-01"
)

jumlah_file_kedua = sum(
    len(files)
    for _, _, files in os.walk(OUT_UJI)
)

print(
    "Jumlah file setelah panggilan kedua:",
    jumlah_file_kedua
)

print(
    "\nIdempoten?",
    hasil_kedua is False
    and jumlah_file_kedua == jumlah_file_pertama
)

=== Panggilan pertama ===
cache ditemukan: yellow_tripdata_2023-01.parquet
tidak ada unduhan ulang
Validasi lulus: 2,986,910 baris x 17 kolom
[SELESAI] 2023-01: 2,986,910 baris
Output: /content/lapisan_terkurasi/trips_bersih_incremental/2023-01
Jumlah file setelah panggilan pertama: 8

=== Panggilan kedua ===
[SKIP] 2023-01 sudah ada di output.
Tidak ada berkas baru yang dibuat.
Jumlah file setelah panggilan kedua: 8

Idempoten? True
